# Chapter 2: Solving the Inference Bottleneck with the Key-Value Cache

## Introduction

The inference bottleneck is one of the most critical challenges in deploying large language models like DeepSeek. This chapter explores how the key-value cache, a fundamental optimization technique addresses this bottleneck and serves as the foundation for more advanced attention mechanisms.

In autoregressive generation, each new token requires attention computations across all previous tokens in the sequence. Without optimizations, this would lead to:

1. Quadratically increasing computation time as sequence length grows
2. Redundant recomputation of key and value tensors for tokens that have already been processed
3. Prohibitive memory and computational costs for practical applications

The key-value cache solves these issues by storing previously computed key-value pairs, dramatically reducing the computational burden during token generation. This foundational technique enables DeepSeek's impressive performance with long contexts of up to 128K tokens.

## 2.1 Understanding Attention Mechanisms

Before diving into the key-value cache, we need to understand the attention mechanism itself. Attention allows a model to focus on relevant parts of the input when producing each element of the output.

### The Attention Formula

The attention mechanism can be expressed mathematically as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ (queries): What we're looking for
- $K$ (keys): What we match against
- $V$ (values): What we extract when we find a match
- $d_k$: The dimension of the keys (used for scaling)

In transformer models like DeepSeek, each attention layer processes:
- **Queries**: Representations of the token we're currently generating
- **Keys/Values**: Representations of all tokens in the context

### Multi-Head Attention

DeepSeek uses multi-head attention, which allows the model to attend to information from different representation subspaces simultaneously:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

Where each head is computed as:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

This creates multiple "attention heads" that can focus on different aspects of the input sequence.

## 2.2 Autoregressive Generation: The Root of the Inference Bottleneck

DeepSeek models, like other transformer-based LLMs, generate text autoregressively—one token at a time, where each new token depends on all previous tokens. This creates a computational challenge during inference:

1. For the first token, we compute attention using just the prompt
2. For the second token, we compute attention using the prompt plus the first generated token
3. For the third token, we compute attention using all previous tokens
4. And so on...

As the sequence grows, each new token requires more computation than the last. Without optimization, this would create:

- **O(n²) complexity** in sequence length for each new token
- **Redundant calculations** as the same keys and values are recomputed for existing tokens
- **Slow inference speed** for practical applications

The code below demonstrates this autoregressive generation process using a simple GPT-2 model. Pay attention to how we generate one token at a time, repeatedly passing the entire sequence through the model:

## Let's Start

### Listing 2.1: Visualizing Autoregressive Generation

The code below demonstrates the token-by-token generation process in an autoregressive model. We start with a prompt and then generate 20 additional tokens sequentially. Notice how:

1. The entire sequence is passed through the model at each step
2. We extract only the logits for the last token position
3. We select the most likely next token (greedy decoding)
4. We append this token to our sequence and repeat

This naive implementation clearly shows why autoregressive generation becomes increasingly slow as the sequence grows longer. Each token requires a full forward pass through all previous tokens.

In [1]:
# =========================================================
# SETUP: Imports and Model Loading for the entire notebook
# =========================================================
import torch
import torch.nn as nn
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Setting up models...")

## Load pre-trained model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("Setup complete.!")


# ==============================================================
# LISTING 2.1: Visualizing autoregressive generation with GPT-2
# ==============================================================
prompt = "The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids

print(f"Prompt: '{prompt}'", end="")

## Generate 20 tokens
for _ in range(20):
    # Pass the entire sequence to the model
    outputs = model(input_ids)
    logits = outputs.logits

    # Get the logits for the very last token
    next_token_logits = logits[:, -1, :]

    # Get the ID of the most likely next token (greedy decoding)
    next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

    # Append the new token ID to the input sequence
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)

    # Decode and print the new token
    new_token = tokenizer.decode(next_token_id[0])
    print(new_token, end="", flush=True)

print("\n")

/media/mo-shwa/shwa-workshop/Artifical Intelligence World/05. LLMs/Workshops/YouTube/Vizuara AI/DeepSeek From Scratch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Setting up models...
Setup complete.!
Prompt: 'The next day is bright' and sunny, and the sun is shining. The sun is shining, and the moon is shining.



### Listing 2.2: Demonstrating the Speedup of KV Caching

The code below demonstrates how **Key-Value (KV) caching) significantly speeds up autoregressive generation**. Instead of passing the entire sequence through the model at every generation step, we cache the **Key and Value tensors** computed for the previous tokens and reuse them. Notice how:

1. **The prompt is processed once** to compute the initial Key and Value tensors.
2. **The previous Key and Value tensors are cached** instead of being recomputed at every step.
3. **Only the newly generated token is passed through the model** during each subsequent step.
4. The new token's **Key and Value tensors are appended to the cache**.
5. The model uses the cached tensors together with the new token's tensors to compute the next prediction.

This dramatically reduces the amount of computation required during generation. Without KV caching, the model repeatedly processes all previous tokens, causing the computation to grow with the sequence length. With KV caching, previously computed Key and Value representations are reused, so each new token only requires computation for that token while still being able to attend to the entire previous context.

In [2]:
# =====================================================
# LISTING 2.2: Demonstrating the Speedup of KV Caching
# =====================================================
prompt = "The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

# -------------------------------
# --- Timing without KV cache ---
# ------------------------------- 
print("↪️ Generating without KV Cache...")
start_time_without_cache = time.time()
output_without_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=False,           # Explicitly disable the cache
    attention_mask=attention_mask
)
end_time_without_cache = time.time()
duration_without_cache = end_time_without_cache - start_time_without_cache
print(f"⏲ Time without KV Cache: {duration_without_cache:.4f} seconds\n")


# ----------------------------
# --- Timing with KV cache ---
# ------------------------------- 
print("🚀 Generating with KV Cache...")
start_time_with_cache = time.time()
output_with_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=True,            # Explicitly enable the cache
    attention_mask=attention_mask
)
end_time_with_cache = time.time()
duration_with_cache = end_time_with_cache - start_time_with_cache
print(f"⏲ Time with KV Cache: {duration_with_cache:.4f} seconds\n")


# --- Calculate and print the speedup ---
speedup = duration_without_cache / duration_with_cache
print(f"KV Cache Speedup: {speedup:.2f}x")

↪️ Generating without KV Cache...


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


⏲ Time without KV Cache: 31.5942 seconds

🚀 Generating with KV Cache...
⏲ Time with KV Cache: 7.2926 seconds

KV Cache Speedup: 4.33x


### 2.3 Key-Value Caching: The First Generation Solution

The key-value (KV) cache is the foundational technique for addressing the inference bottleneck in transformer models like DeepSeek. The key insight is simple but powerful:

**Since previous tokens don't change during generation, their key and value projections can be computed once and reused.**

### How KV Caching Works:

1. **Initial Computation**: For the first token, compute Q, K, V as usual
2. **Storage**: Save the K, V tensors in a "cache" 
3. **Subsequent Tokens**: 
   - Only compute Q, K, V for the new token
   - Retrieve previous K, V from the cache
   - Concatenate the new K, V with the cached ones
   - Store the expanded K, V in the cache
4. **Result**: Each new token only needs to compute its own key-value pair rather than recomputing for all tokens

This reduces the computational complexity from O(n²) to O(n) per token, where n is the sequence length.

### Memory-Computation Tradeoff:

The KV cache trades increased memory usage for significantly reduced computation:

- **Memory Usage**: Increases linearly with sequence length
- **Computation**: Dramatically reduced for long sequences

This is particularly important for DeepSeek models with their 128K context window capability.

### Listing 2.2: Benchmarking KV Cache Performance

The code below demonstrates and measures the dramatic speedup achieved through KV caching:

In [4]:
import threading
import time
import os

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextIteratorStreamer
)


def compare_kv_cache():
    # ==========================================
    # Load model and tokenizer
    # ==========================================
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")

    prompt = "The next day is bright"
    tokens = tokenizer.encode(prompt, return_tensors="pt")

    # ==========================================
    # Streaming function for use in threading
    # ==========================================
    def stream_tokens(use_cache, output_list, timing_list):
        start_time = time.time()

        streamer = TextIteratorStreamer(
            tokenizer,
            skip_special_tokens=True
        )

        generation_kwargs = {
            "input_ids": tokens,
            "max_new_tokens": 50,
            "use_cache": use_cache,
            "streamer": streamer,
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9
        }

        # Start generation in a separate thread
        thread = threading.Thread(
            target=model.generate,
            kwargs=generation_kwargs
        )

        thread.start()

        # Collect tokens as they're generated with timing
        for token in streamer:
            current_time = time.time() - start_time

            timing_list.append(current_time)
            output_list.append(token)

            # Simulate realistic streaming delay
            time.sleep(0.05)

    # ==========================================
    # Output and timing containers
    # ==========================================
    output_with_cache = []
    output_without_cache = []

    timing_with_cache = []
    timing_without_cache = []

    # ==========================================
    # Record overall start time
    # ==========================================
    overall_start = time.time()

    # ==========================================
    # Start both generation processes
    # ==========================================
    thread_with_cache = threading.Thread(
        target=stream_tokens,
        args=(
            True,
            output_with_cache,
            timing_with_cache
        )
    )

    thread_without_cache = threading.Thread(
        target=stream_tokens,
        args=(
            False,
            output_without_cache,
            timing_without_cache
        )
    )

    thread_with_cache.start()
    thread_without_cache.start()

    # Give the threads a little time to start generating
    time.sleep(1)

    # ==========================================
    # Clear screen
    # ==========================================
    os.system("cls" if os.name == "nt" else "clear")

    print("=" * 70)
    print(f"{'KV CACHE STREAMING COMPARISON (Side-by-side)':^70}")
    print("=" * 70)
    print(f"{'WITH KV CACHE':<33} | {'WITHOUT KV CACHE':<33}")
    print("-" * 70)

    # ==========================================
    # Display tokens side-by-side as generated
    # ==========================================
    while (
        thread_with_cache.is_alive()
        or thread_without_cache.is_alive()
        or len(output_with_cache) > 0
        or len(output_without_cache) > 0
    ):

        # Get a token from each output if available
        token_with = (
            output_with_cache.pop(0)
            if output_with_cache
            else ""
        )

        token_without = (
            output_without_cache.pop(0)
            if output_without_cache
            else ""
        )

        if token_with or token_without:
            print(
                f"{token_with:<33} | {token_without:<33}",
                flush=True
            )

            time.sleep(0.05)

        else:
            # If both are empty but threads are still running,
            # wait a bit
            if (
                thread_with_cache.is_alive()
                or thread_without_cache.is_alive()
            ):
                time.sleep(0.1)

            else:
                # If no tokens and threads are done, exit loop
                break

    # ==========================================
    # Wait for threads to complete
    # ==========================================
    thread_with_cache.join()
    thread_without_cache.join()

    total_time = time.time() - overall_start

    # ==========================================
    # Compile full texts
    # ==========================================
    with_cache_text = prompt + " " + "".join(output_with_cache)
    without_cache_text = prompt + " " + "".join(output_without_cache)

    print("=" * 70)

    print(
        f"\n✅ Streaming comparison complete in "
        f"{total_time:.2f} seconds."
    )

    # ==========================================
    # Print timing statistics
    # ==========================================
    print("\n=== ⏲ TIMING COMPARISON ⏲ ===")

    print("\n♻️♻️♻️ Total tokens generated:\n")

    if timing_with_cache:
        print(
            f"- ✔️ With KV Cache: "
            f"{len(timing_with_cache)} tokens in "
            f"{timing_with_cache[-1]:.2f}s"
        )

    if timing_without_cache:
        print(
            f"- ❌ Without KV Cache: "
            f"{len(timing_without_cache)} tokens in "
            f"{timing_without_cache[-1]:.2f}s"
        )

    # ==========================================
    # Calculate average token generation speed
    # ==========================================
    if timing_with_cache:
        with_cache_avg = (
            timing_with_cache[-1]
            / len(timing_with_cache)
        )

        print(
            f"- 📝 With KV Cache avg time per token: "
            f"{with_cache_avg:.4f}s"
        )

    if timing_without_cache:
        without_cache_avg = (
            timing_without_cache[-1]
            / len(timing_without_cache)
        )

        print(
            f"- 📝 Without KV Cache avg time per token: "
            f"{without_cache_avg:.4f}s"
        )

    if timing_with_cache and timing_without_cache:
        speedup = without_cache_avg / with_cache_avg if with_cache_avg > 0 else 0
        print(f"\n🚀 KV Cache Speedup: {speedup:.2f}x Faster")

    # ==========================================
    # Generated Texts
    # ==========================================
    
    print("\n=== 📚 GENERATED TEXTS 📚 ===")
    print("\n🟢 With KV Cache:")
    print(with_cache_text)
    
    print("\n🔴 Without KV Cache:")
    print(without_cache_text)


# Run the comparison
compare_kv_cache()

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


             KV CACHE STREAMING COMPARISON (Side-by-side)             
WITH KV CACHE                     | WITHOUT KV CACHE                 
----------------------------------------------------------------------
The next day is                   | The next day is                  
bright                            |                                  
and                               | bright,                          
sunny                             | and                              
and                               | the                              
I'm                               |                                  
in                                |                                  
the                               |                                  
                                  | sun                              
sun,                              |                                  
                                  | is                               
enjoying          

## 2.4 From KV Cache to Advanced Attention Mechanisms

While the key-value cache dramatically improves inference speed, models like DeepSeek require further optimizations to handle their massive scale efficiently. The KV cache serves as the foundation for more advanced attention mechanisms:

### The Evolution Path:

1. **KV Cache (First Generation)**: The basic optimization we've just explored
2. **Multi-Query Attention (MQA)**: Shares K/V projections across attention heads
3. **Grouped-Query Attention (GQA)**: A middle ground between standard attention and MQA
4. **Multi-Head Latent Attention**: Advanced techniques used in the latest DeepSeek models

Let's explore these evolved attention mechanisms that build on the KV cache foundation.